### This code exports the selected scene to Google Drive.


In [8]:
import ee   
import geemap   
import geopandas as gpd
# ee.Authenticate(auth_mode='localhost')    
ee.Initialize(project='earth-engine-auth-project')     


In [2]:
## true color composite bands
bands_vis_l57 = ['SR_B3', 'SR_B2', 'SR_B1']       # landsat 5,7
bands_vis_l89 = ['SR_B4', 'SR_B3', 'SR_B2']   # landsat 8,9
bands_vis_s2 = ['B4','B3','B2']   
## selected Bands (Landsat 5/7 SR)   
bands_sel_l57 = ['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B7']   
bands_sel_l89 = ['SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7']   
bands_sel_s2 = ['B2','B3','B4','B8','B11','B12']   


In [9]:
path_ygp_vec = 'data/boundary/ygp_region.gpkg'  
ygp_gdf = gpd.read_file(path_ygp_vec) 
ygp_region = geemap.gdf_to_ee(ygp_gdf)   


### check

In [10]:
# Region and image
region = ee.Geometry.Rectangle([107.88, 28.52, 108.53, 29.07])
image = ee.Image('LANDSAT/LC09/C02/T1_L2/LC09_126040_20241030')
bands_sel = bands_sel_l89  # for landsat-8,9
# Clip + select bands + scaling
scene = image.clip(region).select(bands_sel)   # for landsat-8,9
if str(image.id().getInfo()).startswith('L'):    # if Landsat, apply scaling factors
    scene = scene.multiply(0.0000275).add(-0.2).multiply(10000).toInt16() ;  ## for landsat-5789
# Visualization (Jupyter/Colab)
Map = geemap.Map()
Map.addLayer(ygp_region, {}, 'Yunnan-Guizhou Plateau', True, 0.2)
Map.centerObject(region, 9)
Map.addLayer(scene, {'bands': bands_vis_l89, 'min': 0, 'max': 2000}, 'l9_scene_01')
Map


Map(center=[28.795147247688906, 108.20500000000018], controls=(WidgetControl(options=['position', 'transparent…

### download

In [7]:
task = ee.batch.Export.image.toDrive(
    image=scene,
    description='l9_scene_01',
    folder='gee-down',
    scale=30,      ## l5789:30, s2:10
    fileFormat='GeoTIFF',
    region=region)
task.start()
print('Export task started:', task.status())  



Export task started: {'state': 'READY', 'description': 'l9_scene_01', 'priority': 100, 'creation_timestamp_ms': 1780114070183, 'update_timestamp_ms': 1780114070183, 'start_timestamp_ms': 0, 'task_type': 'EXPORT_IMAGE', 'id': '4JFPLCDKTTVOWKNXV3USZQEM', 'name': 'projects/earth-engine-auth-project/operations/4JFPLCDKTTVOWKNXV3USZQEM'}
